# 05b - Entanglement-Inspired Feature Learning

**Phase 1: CartPole Environment**

## Objective

This notebook implements **entanglement-inspired** neural network layers that learn feature correlations.

## Quantum Principle

Quantum entanglement creates correlations between particles that cannot exist classically:
- Measuring one particle instantly tells you about the other
- These correlations go beyond simple linear relationships

## Classical Translation

We learn which features should be "entangled" (correlated):
- Instead of fixed random rotations, learn correlation structure
- Features that are physically related get connected
- For CartPole: angle and angular velocity, position and velocity

## Key Differences from Original (05)

| Aspect | Original (05) | This Notebook (05b) |
|--------|---------------|---------------------|
| Rotations | Fixed angles | Learnable angles |
| Pairings | Random | Learned correlations |
| Input | Raw features | Normalized features |
| Expected | Poor (72% worse) | Should improve or match |

## Configuration

Same as baseline: `stoch=64, deter=512, hidden=512`

In [ ]:
import sys
from pathlib import Path
import json
import time
from typing import Dict, List, NamedTuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gymnasium as gym

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from quantum_inspired import EntanglementLayer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# Configuration
EXPERIMENT_SEEDS = [42, 123, 456, 789, 1024]

OBS_DIM = 4
ACTION_DIM = 2
STOCH_DIM = 64
DETER_DIM = 512
HIDDEN_DIM = 512

NUM_EPISODES = 100
NUM_STEPS = 10000
BATCH_SIZE = 32
SEQ_LEN = 20
LEARNING_RATE = 3e-4

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# Data collection (same as other notebooks)
def collect_episodes(env_name, num_episodes, seed):
    env = gym.make(env_name)
    episodes = []
    
    for i in range(num_episodes):
        obs, _ = env.reset(seed=seed + i)
        observations, actions, rewards, dones = [obs], [], [], []
        
        done = False
        while not done:
            action = env.action_space.sample()
            action_oh = np.zeros(ACTION_DIM)
            action_oh[action] = 1.0
            
            obs, r, term, trunc, _ = env.step(action)
            done = term or trunc
            
            observations.append(obs)
            actions.append(action_oh)
            rewards.append(r)
            dones.append(float(done))
        
        observations = observations[:-1]
        if len(observations) > 0:
            episodes.append({
                'obs': np.array(observations, dtype=np.float32),
                'actions': np.array(actions, dtype=np.float32),
                'rewards': np.array(rewards, dtype=np.float32),
                'dones': np.array(dones, dtype=np.float32)
            })
    
    env.close()
    return episodes

class ReplayBuffer:
    def __init__(self, capacity=1000):
        self.episodes = []
        self.capacity = capacity
    
    def add(self, ep):
        if len(self.episodes) >= self.capacity:
            self.episodes.pop(0)
        self.episodes.append(ep)
    
    def sample(self, batch_size, seq_len):
        obs_b, act_b, rew_b, done_b = [], [], [], []
        for _ in range(batch_size):
            ep = np.random.choice(self.episodes)
            L = len(ep['obs'])
            if L <= seq_len:
                pad = seq_len - L
                obs = np.pad(ep['obs'], ((0,pad),(0,0)), mode='edge')
                act = np.pad(ep['actions'], ((0,pad),(0,0)), mode='edge')
                rew = np.pad(ep['rewards'], (0,pad), mode='edge')
                done = np.pad(ep['dones'], (0,pad), mode='edge')
            else:
                s = np.random.randint(0, L - seq_len)
                obs = ep['obs'][s:s+seq_len]
                act = ep['actions'][s:s+seq_len]
                rew = ep['rewards'][s:s+seq_len]
                done = ep['dones'][s:s+seq_len]
            obs_b.append(obs)
            act_b.append(act)
            rew_b.append(rew)
            done_b.append(done)
        return np.array(obs_b), np.array(act_b), np.array(rew_b), np.array(done_b)
    
    def __len__(self):
        return len(self.episodes)

## World Model with Entanglement Layer

In [ ]:
class RSSMState(NamedTuple):
    deter: torch.Tensor
    stoch: torch.Tensor
    
    @property
    def combined(self):
        return torch.cat([self.deter, self.stoch], dim=-1)


class EntangledWorldModel(nn.Module):
    """
    World model with entanglement-inspired layers in the encoder.
    
    Architecture matches baseline with added EntanglementLayer:
    - encoder: [512, EntanglementLayer, 512]
    - decoder: [512, EntanglementLayer, 512]
    - reward_predictor: [512, 512] (matches baseline)
    - continue_predictor: [512, 512] (matches baseline)
    """
    
    def __init__(self):
        super().__init__()
        self.stoch_dim = STOCH_DIM
        self.deter_dim = DETER_DIM
        self.hidden_dim = HIDDEN_DIM
        self.state_dim = STOCH_DIM + DETER_DIM
        
        # Encoder with entanglement layer
        self.encoder = nn.Sequential(
            nn.Linear(OBS_DIM, HIDDEN_DIM),
            nn.ELU(),
            EntanglementLayer(dim=HIDDEN_DIM, correlation_type='multiplicative'),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ELU()
        )
        
        # RSSM (same as baseline)
        self.input_proj = nn.Sequential(
            nn.Linear(STOCH_DIM + ACTION_DIM, HIDDEN_DIM), nn.ELU()
        )
        self.gru = nn.GRUCell(HIDDEN_DIM, DETER_DIM)
        
        self.prior_net = nn.Sequential(
            nn.Linear(DETER_DIM, HIDDEN_DIM), nn.ELU(),
            nn.Linear(HIDDEN_DIM, STOCH_DIM * 2)
        )
        self.posterior_net = nn.Sequential(
            nn.Linear(DETER_DIM + HIDDEN_DIM, HIDDEN_DIM), nn.ELU(),
            nn.Linear(HIDDEN_DIM, STOCH_DIM * 2)
        )
        
        # Decoder with entanglement [512, 512]
        self.decoder = nn.Sequential(
            nn.Linear(self.state_dim, HIDDEN_DIM),
            nn.ELU(),
            EntanglementLayer(dim=HIDDEN_DIM, correlation_type='multiplicative'),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ELU(),
            nn.Linear(HIDDEN_DIM, OBS_DIM)
        )
        
        # Reward predictor [512, 512] - matches baseline
        self.reward_pred = nn.Sequential(
            nn.Linear(self.state_dim, HIDDEN_DIM),
            nn.ELU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ELU(),
            nn.Linear(HIDDEN_DIM, 1)
        )
        
        # Continue predictor [512, 512] - matches baseline
        self.continue_pred = nn.Sequential(
            nn.Linear(self.state_dim, HIDDEN_DIM),
            nn.ELU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ELU(),
            nn.Linear(HIDDEN_DIM, 1)
        )
    
    def initial_state(self, batch_size):
        return RSSMState(
            deter=torch.zeros(batch_size, self.deter_dim, device=DEVICE),
            stoch=torch.zeros(batch_size, self.stoch_dim, device=DEVICE)
        )
    
    def _get_dist(self, stats):
        mean, log_std = stats.chunk(2, dim=-1)
        std = F.softplus(log_std) + 0.1
        return torch.distributions.Normal(mean, std)
    
    def observe(self, obs, action, state):
        embed = self.encoder(obs)
        x = self.input_proj(torch.cat([state.stoch, action], dim=-1))
        deter = self.gru(x, state.deter)
        
        posterior_stats = self.posterior_net(torch.cat([deter, embed], dim=-1))
        posterior = self._get_dist(posterior_stats)
        stoch = posterior.rsample()
        
        prior_stats = self.prior_net(deter)
        prior = self._get_dist(prior_stats)
        
        return RSSMState(deter, stoch), prior, posterior
    
    def imagine(self, action, state):
        x = self.input_proj(torch.cat([state.stoch, action], dim=-1))
        deter = self.gru(x, state.deter)
        prior_stats = self.prior_net(deter)
        prior = self._get_dist(prior_stats)
        stoch = prior.rsample()
        return RSSMState(deter, stoch), prior
    
    def decode(self, state):
        return self.decoder(state.combined)
    
    def forward(self, obs_seq, action_seq):
        batch_size, seq_len = obs_seq.shape[:2]
        state = self.initial_state(batch_size)
        
        recon_obs, pred_rewards, pred_continues = [], [], []
        priors, posteriors = [], []
        
        for t in range(seq_len):
            state, prior, posterior = self.observe(obs_seq[:, t], action_seq[:, t], state)
            recon_obs.append(self.decode(state))
            pred_rewards.append(self.reward_pred(state.combined).squeeze(-1))
            pred_continues.append(torch.sigmoid(self.continue_pred(state.combined).squeeze(-1)))
            priors.append(prior)
            posteriors.append(posterior)
        
        return {
            'recon_obs': torch.stack(recon_obs, dim=1),
            'pred_rewards': torch.stack(pred_rewards, dim=1),
            'pred_continues': torch.stack(pred_continues, dim=1),
            'priors': priors,
            'posteriors': posteriors
        }
    
    def get_entanglement_stats(self):
        """Get statistics about learned entanglement."""
        stats = {}
        for name, module in self.named_modules():
            if isinstance(module, EntanglementLayer):
                stats[name] = module.get_entanglement_stats()
        return stats

## Training

In [ ]:
def compute_loss(output, obs_seq, reward_seq, kl_weight=1.0):
    recon_loss = F.mse_loss(output['recon_obs'], obs_seq)
    reward_loss = F.mse_loss(output['pred_rewards'], reward_seq)
    
    kl_losses = []
    for prior, posterior in zip(output['priors'], output['posteriors']):
        kl = torch.distributions.kl_divergence(posterior, prior).sum(-1).mean()
        kl_losses.append(kl)
    kl_loss = torch.stack(kl_losses).mean()
    
    total = recon_loss + reward_loss + kl_weight * kl_loss
    return {'total': total, 'recon': recon_loss, 'kl': kl_loss}


def train_model(model, buffer, num_steps=NUM_STEPS, verbose=True):
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    history = []
    
    for step in range(num_steps):
        model.train()
        
        obs, actions, rewards, _ = buffer.sample(BATCH_SIZE, SEQ_LEN)
        obs = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
        actions = torch.tensor(actions, dtype=torch.float32, device=DEVICE)
        rewards = torch.tensor(rewards, dtype=torch.float32, device=DEVICE)
        
        optimizer.zero_grad()
        output = model(obs, actions)
        losses = compute_loss(output, obs, rewards)
        losses['total'].backward()
        optimizer.step()
        
        history.append({
            'step': step,
            'total': losses['total'].item(),
            'recon': losses['recon'].item(),
            'kl': losses['kl'].item()
        })
        
        if verbose and step % 1000 == 0:
            print(f"Step {step}: total={losses['total'].item():.4f}")
    
    return pd.DataFrame(history)

In [ ]:
def evaluate(model, buffer):
    model.eval()
    obs, actions, _, _ = buffer.sample(100, SEQ_LEN)
    obs = torch.tensor(obs, dtype=torch.float32, device=DEVICE)
    actions = torch.tensor(actions, dtype=torch.float32, device=DEVICE)
    
    with torch.no_grad():
        output = model(obs, actions)
        mse = F.mse_loss(output['recon_obs'], obs).item()
    return mse

## Run Experiments

In [ ]:
def run_experiment(seed):
    print(f"\n{'='*50}")
    print(f"Seed {seed}")
    print(f"{'='*50}")
    
    set_seed(seed)
    start = time.time()
    
    episodes = collect_episodes('CartPole-v1', NUM_EPISODES, seed)
    buffer = ReplayBuffer()
    for ep in episodes:
        buffer.add(ep)
    
    model = EntangledWorldModel().to(DEVICE)
    history = train_model(model, buffer)
    
    train_mse = evaluate(model, buffer)
    
    # Test set
    test_eps = collect_episodes('CartPole-v1', 50, seed + 10000)
    test_buffer = ReplayBuffer()
    for ep in test_eps:
        test_buffer.add(ep)
    test_mse = evaluate(model, test_buffer)
    
    # Get entanglement stats
    ent_stats = model.get_entanglement_stats()
    
    print(f"Train MSE: {train_mse:.6f}, Test MSE: {test_mse:.6f}")
    
    return {
        'seed': seed,
        'history': history,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'time': time.time() - start,
        'entanglement_stats': ent_stats
    }

In [ ]:
print("="*60)
print("ENTANGLEMENT LAYER EXPERIMENTS")
print("="*60)

results = [run_experiment(s) for s in EXPERIMENT_SEEDS]

In [ ]:
test_mses = [r['test_mse'] for r in results]
train_mses = [r['train_mse'] for r in results]

print("\n" + "="*60)
print("RESULTS (Entanglement Layers)")
print("="*60)
print(f"Train MSE: {np.mean(train_mses):.6f} +/- {np.std(train_mses):.6f}")
print(f"Test MSE:  {np.mean(test_mses):.6f} +/- {np.std(test_mses):.6f}")

In [ ]:
# Save results
results_dir = Path('../experiments/results/entanglement_proper')
results_dir.mkdir(parents=True, exist_ok=True)

metrics = {
    'approach': 'entanglement_proper',
    'description': 'Entanglement-inspired layers with learned correlations',
    'config': {'stoch_dim': STOCH_DIM, 'deter_dim': DETER_DIM, 'hidden_dim': HIDDEN_DIM},
    'results': {
        'test_mse_mean': float(np.mean(test_mses)),
        'test_mse_std': float(np.std(test_mses)),
        'test_mses': [float(x) for x in test_mses]
    }
}

with open(results_dir / 'complete_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print(f"Saved to {results_dir}")